# Clean2 v6 — Text-only LLM Audit (Hard Negative Aware)

**Pipeline:** GPT relabel + remove → apply → dataset_relabeled.jsonl

**Changes from v5:**
- ❌ No Gemini voice mismatch (ตัด E2E ออก)
- ❌ No formal_score (ไม่ใช้แล้ว)
- ✅ Hard negative aware prompt (ไม่ลบ real_otp / friend_new_number / etc.)
- ✅ Uses dataset.jsonl (มี subtype field → วัด noise ratio ได้)
- ✅ OpenRouter (bypass OpenAI tier limit)

**ลำดับรัน:** Cell 1 → 2 → 3 → 4 → 5 → 6 → (ถ้า noise < 20%: MODE=all + APPLY=True → 4 → 7 → 8)


In [ ]:
# === Cell 1: Setup ===
!pip install -q openai

from google.colab import drive, userdata, files
drive.mount('/content/drive')

import os, json, time, re, zipfile, random, threading
import concurrent.futures as cf
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

DRIVE_BASE = '/content/drive/MyDrive/ThaiScamCall'
OUT_DIR    = f'{DRIVE_BASE}/clean'
ZIP_AUDIO  = f'{DRIVE_BASE}/mp3_15s.zip'
EXT_AUDIO  = '/content/data_15s'
os.makedirs(OUT_DIR, exist_ok=True)

# OpenRouter client
gpt = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=userdata.get('OPENROUTER_KEY')
)
GPT_MODEL = 'openai/gpt-4o-mini'   # 🔧 ลอง: openai/gpt-5-mini, anthropic/claude-sonnet-4.6

# ===== Config =====
MODE          = 'sample'   # 🔧 'sample'(300,~$0.05) | 'all'(21K,~$2-3)
SAMPLE_N      = 300
GPT_WORKERS   = 20
PROMPT_VER    = 'v6'
REALISTIC_MIN = 3

print(f'✅ Setup OK | MODE={MODE} | model={GPT_MODEL} | prompt={PROMPT_VER}')


In [ ]:
# === Cell 2: Upload dataset_cleaned.jsonl + dataset.jsonl (สำหรับ subtype map) ===
def full_text(d):
    return ' '.join(t['text'] for t in d['turns'])

# --- Step 1: Upload dataset_cleaned.jsonl (ลบ duplicate แล้ว ~21K) ---
print('📤 (1/2) Upload dataset_cleaned.jsonl (ลบ duplicate แล้ว)')
up1 = files.upload()
fname1 = list(up1.keys())[0]

data = []
for line in up1[fname1].decode('utf-8').splitlines():
    if line.strip(): data.append(json.loads(line))

for i, d in enumerate(data):
    d['line']    = i + 1
    d['conv_id'] = f'conv_{i+1:05d}_label{d["label"]}'

print(f'\n✅ Cleaned: {len(data)} บท | Keys: {list(data[0].keys())}')

# --- Step 2: Upload dataset.jsonl (มี subtype — สำหรับ map เข้า data) ---
has_subtype = 'subtype' in data[0]
if has_subtype:
    print('\n✅ dataset_cleaned มี subtype field อยู่แล้ว — ข้ามขั้นตอนนี้')
else:
    print('\n📤 (2/2) Upload dataset.jsonl (มี subtype — สำหรับ map)')
    up2 = files.upload()
    fname2 = list(up2.keys())[0]

    orig = []
    for ln in up2[fname2].decode('utf-8').splitlines():
        if ln.strip(): orig.append(json.loads(ln))
    print(f'   Original: {len(orig)} บท')

    # build map: full_text → subtype
    orig_map = {}
    for d in orig:
        key = full_text(d)
        if key not in orig_map:
            orig_map[key] = d.get('subtype', 'NONE')

    matched = 0
    for d in data:
        sub = orig_map.get(full_text(d), 'NONE')
        d['subtype'] = sub
        if sub != 'NONE': matched += 1
    print(f'   ✅ matched subtype: {matched}/{len(data)} ({matched/len(data)*100:.1f}%)')

# --- สรุป ---
subs = Counter(d.get('subtype', 'NONE') for d in data)
print(f'\nSubtype distribution (top 10):')
for s, c in subs.most_common(10): print(f'  {s}: {c}')

print(f'\nLabel:')
print(pd.Series([d['label'] for d in data]).value_counts()
      .rename({0:'not_scam', 1:'scam'}))


In [ ]:
# === Cell 3: GPT check function (prompt v6 — hard negative aware) ===
SYSTEM = """คุณตรวจสอบบทสนทนาโทรศัพท์ (ฝึก AI ตรวจ scam call ไทย) จาก transcript
label เดิม: 0=ปกติ(not_scam), 1=scam — เชื่อ label เดิมไว้ก่อน

ตอบ JSON:
{
  "true_label": 0 หรือ 1,
  "realistic": 0-10,
  "action": "keep"/"relabel"/"remove",
  "reason": "สั้นๆ"
}

⚠️ relabel asymmetric: ก้ำกึ่ง → keep (default)

🎯 SCAM ชัด (relabel 0→1 ได้):
- ตำรวจ/สรรพากร/ธนาคาร ขู่/อายัด/คดี/ฟอกเงิน → โอนเงิน
- กดลิงก์/ติดตั้ง AnyDesk/TeamViewer/USSD
- ขอ Line ID @police-xxx
- ลงทุนผลตอบแทน 30-50%/เดือน
- งานออนไลน์รายได้ดีเกินจริง + ฝากเงินปลดล็อก

✅ HARD NEGATIVE — label=0 ⚠️ ห้าม relabel เป็น scam!:

(A) real_otp: ธนาคารโทรกลับ confirm OTP สำหรับ transaction
    ที่ user เริ่มเอง — มี reference number ชัด
    ≠ scammer ขอ OTP สุ่ม (= scam)

(B) friend_new_number: เพื่อนจริงเปลี่ยนเบอร์ + ยืมเงิน
    - มี personal context (อ้างชื่อ, ความสัมพันธ์, เหตุผลปกติ)
    - ไม่กดดัน, ยอมให้ verify, จำนวนเล็กน้อย
    ≠ "เปลี่ยนเบอร์ + ขอด่วน + vague" (= scam)

(C) friend_borrow / colleague_expense / family_help:
    ยืมเงินจริง — มี context ชัดเจน, ไม่กดดัน

(D) verification (otp_thai/english/reset_code):
    ระบบส่ง OTP ให้ user (เช่น "รหัสคือ 123456 ห้ามเปิดเผย")

(E) official (bank_real/gov_notice/hospital):
    ธนาคาร/หน่วยงานจริง — ไม่ขอ OTP, ไม่กดลิงก์

(F) delivery / legit_promo (AIS/dtac/ประกัน):
    ไม่ขอเงิน-OTP-คลิกลิงก์ = ปกติ

หลักการ:
- ก้ำกึ่ง → keep label เดิม
- "ขอ OTP" → ดู context: มี reference + user เริ่ม transaction = real
- "เพื่อนยืมเงิน" → มี personal detail + ไม่กดดัน = real

action=remove = บทเสีย (breathing/instruction หลุด/สั้นเกิน)
realistic ต่ำ = ฉากไม่น่าเกิดจริง"""

def gpt_check(d):
    txt = '\n'.join(f"{t['speaker']}: {t['text']}" for t in d['turns'])
    for attempt in range(4):
        try:
            r = gpt.chat.completions.create(
                model=GPT_MODEL,
                messages=[{'role':'system','content':SYSTEM},
                          {'role':'user','content':f'label เดิม={d["label"]}\nบทสนทนา:\n{txt}'}],
                temperature=0, max_tokens=120,
                response_format={'type':'json_object'})
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            if attempt == 3:
                return {'action':'error', 'reason':str(e)[:60]}
            time.sleep(2 ** attempt)

print('Test:', gpt_check(data[0]))


In [ ]:
# === Cell 4: Run GPT (parallel + checkpoint + resume errors) ===
target = (random.Random(42).sample(data, min(SAMPLE_N, len(data)))
          if MODE == 'sample' else data)

CKPT = f'{OUT_DIR}/gpt_ckpt_{MODE}_{PROMPT_VER}.jsonl'
done = {}
if os.path.exists(CKPT):
    for ln in open(CKPT, encoding='utf-8'):
        if ln.strip():
            o = json.loads(ln); done[o['conv_id']] = o

todo = [d for d in target
        if d['conv_id'] not in done or done[d['conv_id']].get('action') == 'error']
print(f'MODE={MODE} | target={len(target)} | done={len(done)} | todo={len(todo)}')

lock = threading.Lock()
ckf  = open(CKPT, 'a', encoding='utf-8')
def work(d):
    r = gpt_check(d)
    rec = {'conv_id':d['conv_id'],
           'true_label':r.get('true_label', d['label']),
           'realistic':r.get('realistic', 5),
           'action':r.get('action', 'keep'),
           'reason':r.get('reason', '')}
    with lock:
        ckf.write(json.dumps(rec, ensure_ascii=False) + '\n'); ckf.flush()
    return rec
with cf.ThreadPoolExecutor(GPT_WORKERS) as ex:
    list(tqdm(ex.map(work, todo), total=len(todo), desc='GPT'))
ckf.close()

# merge
done = {}
for ln in open(CKPT, encoding='utf-8'):
    if ln.strip():
        o = json.loads(ln); done[o['conv_id']] = o
for d in target:
    o = done.get(d['conv_id'], {})
    d['gpt_label'] = o.get('true_label', d['label'])
    d['realistic'] = o.get('realistic', 5)
    d['action']    = o.get('action', 'keep')
    d['reason']    = o.get('reason', '')

# สรุป
acts = Counter(d['action'] for d in target)
up   = sum(d['action']=='relabel' and d['label']==0 and d['gpt_label']==1 for d in target)
down = sum(d['action']=='relabel' and d['label']==1 and d['gpt_label']==0 for d in target)
low  = sum(d['realistic'] < REALISTIC_MIN for d in target)
print('\n=== Action ===')
for a,c in acts.items(): print(f'  {a}: {c} ({c/len(target)*100:.1f}%)')
print(f'  relabel 0→1: {up} | 1→0: {down}')
print(f'  realistic<{REALISTIC_MIN}: {low}')
print(f'  errors: {acts.get("error",0)}')


In [ ]:
# === Cell 5: Inspect + save QA CSV ===
df = pd.DataFrame([{
    'conv_id':d['conv_id'], 'label':d['label'], 'gpt_label':d['gpt_label'],
    'subtype':d.get('subtype','NONE'),
    'realistic':d['realistic'],
    'action':d['action'], 'reason':d['reason'], 'text':full_text(d)[:90],
} for d in target])

print('=== RELABEL ===')
display(df[df.action=='relabel'][['conv_id','subtype','label','gpt_label','reason','text']].head(20))
print('=== REMOVE ===')
display(df[df.action=='remove'][['conv_id','reason','text']].head(10))

# QA CSV
rl = df[df.action=='relabel'].copy()
rl['full_text'] = [full_text(d) for d in target if d['action']=='relabel']
rl['correct(T/F)'] = ''
rl[['conv_id','subtype','label','gpt_label','reason','full_text','correct(T/F)']].to_csv(
    f'{OUT_DIR}/relabels_review_{MODE}.csv', index=False, encoding='utf-8-sig')
print(f'\n💾 saved relabels_review_{MODE}.csv ({len(rl)} ตัว)')


In [ ]:
# === Cell 6: วัด Hard Negative noise ratio (สำคัญ!) ===
PROTECTED_SUBTYPES = {
    'real_otp','real_job','friend_borrow','real_parcel','friend_new_number',
    'colleague_expense','family_help',
    'otp_thai','otp_english','reset_code',
    'bank_real','gov_notice','hospital','company_hr',
    'kerry_courier','food_delivery','ems_post','lazada_shopee',
    'ais_promo','true_promo','dtac_promo','bank_offer','insurance_offer',
}

hn_in_sample = [d for d in target if d.get('subtype') in PROTECTED_SUBTYPES]
hn_wrong = [d for d in hn_in_sample
            if d['action']=='relabel' and d['label']==0 and d['gpt_label']==1]
total_relabel = [d for d in target if d['action']=='relabel']
noise_ratio = len(hn_wrong)/max(len(total_relabel),1)*100

print(f'📊 Hard Negative Test ({PROMPT_VER} + {GPT_MODEL})')
print(f'   HN ใน sample:        {len(hn_in_sample)}')
print(f'   HN ถูก relabel ผิด:   {len(hn_wrong)}')
print(f'   Total relabel:       {len(total_relabel)}')
print(f'   ❗ Noise ratio:       {noise_ratio:.1f}%')

if noise_ratio < 20:    print('\n✅ ผ่าน! Run full ได้')
elif noise_ratio < 40:  print('\n⚠️ ลอง model: openai/gpt-5-mini')
else:                   print('\n❌ ลอง model: anthropic/claude-sonnet-4.6')

print('\n=== HN ที่ผิด ===')
for d in hn_wrong[:10]:
    print(f"[{d['conv_id']}] {d.get('subtype')}: {d['reason'][:60]}")

wrong_subs = Counter(d.get('subtype') for d in hn_wrong)
print('\n=== Subtype ผิดเยอะสุด ===')
for s, c in wrong_subs.most_common(5):
    print(f'  {s}: {c}')


In [ ]:
# === Cell 7: Apply → dataset_relabeled.jsonl + mp3_15s_clean.zip ===
APPLY = False   # 🔧 True เมื่อ MODE='all' + ตรวจผลแล้ว

if APPLY and MODE == 'all':
    clean = []; n_rm = 0; n_rl = 0; rm_lines = set()
    for d in data:
        if d['action'] == 'remove' or d['realistic'] < REALISTIC_MIN:
            n_rm += 1; rm_lines.add(d['line']); continue
        lbl = d['gpt_label'] if d['action'] == 'relabel' else d['label']
        if d['action'] == 'relabel': n_rl += 1
        clean.append({'label': lbl, 'turns': d['turns']})

    local_jsonl = 'dataset_relabeled.jsonl'
    with open(local_jsonl, 'w', encoding='utf-8') as f:
        for d in clean: f.write(json.dumps(d, ensure_ascii=False) + '\n')
    !cp {local_jsonl} {OUT_DIR}/{local_jsonl}

    if os.path.exists(ZIP_AUDIO):
        if not os.path.exists(EXT_AUDIO) or len(os.listdir(EXT_AUDIO)) < 1000:
            print('Extracting audio...')
            with zipfile.ZipFile(ZIP_AUDIO) as z: z.extractall(EXT_AUDIO)
        keep_au = [f for f in os.listdir(EXT_AUDIO) if f.endswith('.mp3')
                   and int(re.search(r'conv_(\d+)_', f).group(1)) not in rm_lines]
        local_zip = 'mp3_15s_clean.zip'
        with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fn in tqdm(keep_au, desc='zip'): zf.write(f'{EXT_AUDIO}/{fn}', arcname=fn)
        !cp {local_zip} {DRIVE_BASE}/{local_zip}
        print(f'✅ mp3_15s_clean.zip: {len(keep_au)} ไฟล์')

    print(f'✅ dataset_relabeled.jsonl: {len(clean)} บท (remove {n_rm}, relabel {n_rl})')
else:
    print('⏸️ ตั้ง MODE=all + APPLY=True')


In [ ]:
# === Cell 8: Report ===
hn_wrong_n = sum(1 for d in target
    if d.get('subtype') in PROTECTED_SUBTYPES
    and d['action']=='relabel' and d['label']==0 and d['gpt_label']==1)
total_rl = sum(1 for d in target if d['action']=='relabel')

report = {
    'mode': MODE,
    'model': GPT_MODEL,
    'prompt_version': PROMPT_VER,
    'n_total': len(target),
    'keep': sum(d['action']=='keep' for d in target),
    'remove_broken': sum(d['action']=='remove' for d in target),
    'remove_unrealistic': sum(d['action']!='remove' and d['realistic']<REALISTIC_MIN for d in target),
    'relabel': total_rl,
    'relabel_0to1': sum(d['action']=='relabel' and d['label']==0 and d['gpt_label']==1 for d in target),
    'relabel_1to0': sum(d['action']=='relabel' and d['label']==1 and d['gpt_label']==0 for d in target),
    'hard_negative_wrong': hn_wrong_n,
    'hard_negative_noise_pct': round(hn_wrong_n/max(total_rl,1)*100, 2),
}
with open(f'{OUT_DIR}/clean_report_{MODE}.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))
